# 22 · Opto − Hard-A / Hard-B + adaptation

The same PPC-silencing analysis as notebook 20 − within-phase (opto/masking
sessions, {toi} vs non_opto), between-phase (opto vs masking, all trials),
delta-of-deltas, and the WT-vs-HET population fold − but on a **hard**
distribution. Set `DIST` below to `'Hard-A'` or `'Hard-B'` and re-run.

On top, **§5 adds the adaptation view**: within-session rolling trajectories
after the switch (session by session, since the curriculum is A B A B A B),
opto vs masking sessions, and a 2x2 sliced by genotype (top) and manipulation
(bottom).

**Adaptation caveat.** 30% interleaved silencing is weak for cumulative updating
− the rolling trace inherits a shared running average, so silencing 30% only
weakly perturbs it. Opto-vs-masking adaptation differences are **diluted, and a
null is not strong evidence**. Descriptive trajectories, not a powered test.

In [ ]:
from shared_setup import *
from sound_categorisation.plotting.opto import plot_delta_swarm
apply_style()

experiment, info = load_data()
by_animal, groups = gather_genotypes(experiment)
het_ids, wt_ids = groups.get('het', []), groups.get('wt', [])
opto_ids = [a for a in OPTO_COHORT if a in experiment.animals]
print(f"opto cohort ({len(opto_ids)}): het={[a for a in opto_ids if by_animal.get(a)=='het']} "
      f"wt={[a for a in opto_ids if by_animal.get(a)=='wt']}")

In [ ]:
DIST = 'Hard-A'          # set 'Hard-B' and re-run for the other hard distribution
# 'psychometric' expands to mu, sigma, lapse_low, lapse_high
STATS = ['accuracy', 'hard_accuracy', 'easy_accuracy', 'recency', 'side_bias', 'psychometric']
SENSITIVITY = ['accuracy', 'hard_accuracy', 'easy_accuracy', 'sigma', 'recency', 'lapse_low', 'lapse_high']
BIAS        = ['mu', 'side_bias']          # the light-artifact channel
DISPLAY     = SENSITIVITY + BIAS

# psychometric re-fits per resample -> minutes/animal at 2000; drop to 1000 to iterate
N_PERM, N_BOOT = 10, 10

def rows_from(diffs, aid):
    return collect_rows([{'stat': k, 'value': float(v)} for k, v in diffs.items()],
                        animal=aid, group=by_animal[aid])

## §2 · Single animal, end to end

**SS16 (HET)** through the pipeline for all three contrasts. The loop in §3 is
exactly these calls.

In [ ]:
aid = 'SS16'
animal = experiment.get_animal(aid)
op = select_sessions(animal, distribution=DIST, session_type='opto')
mk = select_sessions(animal, distribution=DIST, session_type='masking')

opto_on,  opto_off = filter_trials(op, trial_type='opto'), filter_trials(op, trial_type='non_opto')
mask_on,  mask_off = filter_trials(mk, trial_type='opto'), filter_trials(mk, trial_type='non_opto')
opto_all, mask_all = filter_trials(op, trial_type='all'),  filter_trials(mk, trial_type='all')

within_opto    = compute_delta_stat({'on': opto_on, 'off': opto_off}, stats=STATS,
                               reference='off', n_permutations=N_PERM, n_bootstrap=N_BOOT, resample_units=('trials', 'sessions'))   # within-phase effect
within_masking    = compute_delta_stat({'on': mask_on, 'off': mask_off}, stats=STATS,
                               reference='off', n_permutations=N_PERM, n_bootstrap=N_BOOT, resample_units=('trials', 'sessions'))
between_opto_masking = compute_delta_stat({'opto': opto_all, 'masking': mask_all}, stats=STATS,
                               reference='masking', n_permutations=0, n_bootstrap=N_BOOT, resample_units=('trials', 'sessions'))     # between-phase difference
delta_of_deltas     = compute_interaction(within_opto, within_masking, contrast='on_vs_off')                        # delta-of-deltas

In [ ]:
print(f"{aid} ({by_animal[aid]}) — {animal.n_sessions} sessions")
print(animal.session_table.groupby(['distribution', 'session_type']).size().to_string())
for name, s in [('opto', op), ('masking', mk)]:
    print(f"  {name}: {pool_arrays(filter_trials(s, trial_type='all'))['n_trials']} trials")

In [ ]:
# psychometric overlays — on vs off (opto, masking)
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (title, on, off) in zip(axes, [('opto', opto_on, opto_off), ('masking', mask_on, mask_off)]):
    plot_psychometric(compute_psychometric(off), ax=ax, color='0.55', label='off')
    plot_psychometric(compute_psychometric(on),  ax=ax, color=get_colour(3), label='on')
    ax.set_title(f"{aid} · {by_animal[aid]} · {DIST} · {title}: on vs off", fontsize=9); ax.legend(frameon=False)
fig.tight_layout()

In [ ]:
# within-phase effect — opto on vs off, per stat
fig, axes = plt.subplots(3, 3, figsize=(10, 9)); axf = axes.ravel()
for ax, stat in zip(axf, DISPLAY): plot_stat_comparison_single(within_opto, stat, ax=ax, units=('trials',))
for ax in axf[len(DISPLAY):]: fig.delaxes(ax)
fig.suptitle(f"{aid} · {by_animal[aid]} · {DIST} · opto on − off trials  (within-phase effect)", fontsize=12); fig.tight_layout()

In [ ]:
# between-phase difference — opto session vs masking session, all trials, per stat
fig, axes = plt.subplots(3, 3, figsize=(10, 9)); axf = axes.ravel()
for ax, stat in zip(axf, DISPLAY): plot_stat_comparison_single(between_opto_masking, stat, ax=ax, units=('trials', 'sessions'))
for ax in axf[len(DISPLAY):]: fig.delaxes(ax)
fig.suptitle(f"{aid} · {by_animal[aid]} · {DIST} · opto vs masking sessions, all trials  (between-phase difference)", fontsize=12); fig.tight_layout()

In [ ]:
# delta-of-deltas — interaction: opto-Δ vs masking-Δ vs their difference (silencing beyond artifact)
fig, axes = plt.subplots(3, 3, figsize=(11, 9)); axf = axes.ravel()
for ax, stat in zip(axf, DISPLAY): plot_interaction_single(delta_of_deltas, stat, ax=ax, units=('trials', 'sessions'))
for ax in axf[len(DISPLAY):]: fig.delaxes(ax)
fig.suptitle(f"{aid} · {by_animal[aid]} · {DIST} · silencing isolated: (opto on−off) − (masking on−off)  (delta-of-deltas)", fontsize=12); fig.tight_layout()

In [ ]:
# update matrices — on vs off, opto and masking (descriptive; order-dependent, not folded)
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for row, (name, on, off) in enumerate([('opto', opto_on, opto_off), ('masking', mask_on, mask_off)]):
    plot_um(compute_um(on),  ax=axes[row, 0]); axes[row, 0].set_title(f"{name} on")
    plot_um(compute_um(off), ax=axes[row, 1]); axes[row, 1].set_title(f"{name} off")
fig.tight_layout()
fig.suptitle(f"{aid} · {by_animal[aid]} · {DIST} · update matrices (opto/masking, on/off)", fontsize=12)

## §3 · All animals — thin fold

One loop, body = the §2 pipeline. Emits three per-animal `{animal, group, stat,
value}` sets: **within-phase** (`within_df`), **between-phase** (`between_df`), **delta-of-deltas** (`delta_of_deltas_df`).

In [ ]:
within_rows, between_rows, delta_of_deltas_rows, per_animal = [], [], [], {}
for aid in opto_ids:
    animal = experiment.get_animal(aid)
    op = select_sessions(animal, distribution=DIST, session_type='opto')
    mk = select_sessions(animal, distribution=DIST, session_type='masking')
    oon, ooff = filter_trials(op, trial_type='opto'), filter_trials(op, trial_type='non_opto')
    if not (oon and ooff):
        print(f"skip {aid}: opto on/off missing"); continue
    within_opto = compute_delta_stat({'on': oon, 'off': ooff}, stats=STATS,
                                reference='off', n_permutations=N_PERM, n_bootstrap=N_BOOT)
    within_rows += rows_from(within_opto['contrasts']['on_vs_off']['diffs'], aid)
    per_animal[aid] = within_opto
    o_all, m_all = filter_trials(op, trial_type='all'), filter_trials(mk, trial_type='all')
    if not (o_all and m_all):
        print(f"note {aid}: no masking — between-phase / delta-of-deltas skipped"); continue
    between_opto_masking = compute_delta_stat({'opto': o_all, 'masking': m_all}, stats=STATS,
                                   reference='masking', n_permutations=0, n_bootstrap=N_BOOT)
    between_rows += rows_from(between_opto_masking['contrasts']['opto_vs_masking']['diffs'], aid)
    mon, moff = filter_trials(mk, trial_type='opto'), filter_trials(mk, trial_type='non_opto')
    if mon and moff:
        within_masking = compute_delta_stat({'on': mon, 'off': moff}, stats=STATS,
                                    reference='off', n_permutations=N_PERM, n_bootstrap=N_BOOT)
        delta_of_deltas = compute_interaction(within_opto, within_masking, contrast='on_vs_off')
        delta_of_deltas_rows += collect_rows([{'stat': s, 'value': float(delta_of_deltas[s]['interaction'])}
                                    for s in delta_of_deltas if s != 'meta'], animal=aid, group=by_animal[aid])

within_df       = pd.DataFrame(within_rows)
between_df     = pd.DataFrame(between_rows)
delta_of_deltas_df = pd.DataFrame(delta_of_deltas_rows)
print(f"within: {within_df['animal'].nunique()} | between: {between_df['animal'].nunique() if len(between_df) else 0} "
      f"| delta_of_deltas: {delta_of_deltas_df['animal'].nunique() if len(delta_of_deltas_df) else 0} animals")

## §4 · Population — WT vs HET

Unit = animals. Effect size first; `p`/`min_p` a map (smallest two-sided p ≈ 0.016
at this n). Each contrast folded WT vs HET. Read the **bias** stats (side_bias, mu)
as the light-artifact channel; the **sensitivity** stats as the performance channel
spared by the artifact.

**On p-values:** the within-phase effect is permutation-valid (laser randomised per trial). between-phase and delta-of-deltas are
**bootstrap only** — session type isn't randomised per trial, so there is no valid
within-animal permutation p; read their per-animal effect from the bootstrap CI. The
p annotated on every swarm below is the *genotype* WT-vs-HET rank test, which is
valid for all three.

In [ ]:
import math
def genotype_table(df):
    res = compare_groups(df, group_col='group')
    tbl = pd.DataFrame([{'stat': s, 'median_wt': r.get('median_a'), 'median_het': r.get('median_b'),
                         'p': r.get('p'), 'min_p': r.get('min_p')} for s, r in res.items()]
                       ).set_index('stat').reindex(DISPLAY).round(4)
    return res, tbl

def swarm_grid(df, res, title):
    ncols = 4; nrows = math.ceil(len(DISPLAY) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.3 * ncols, 3.3 * nrows), squeeze=False); axf = axes.ravel()
    for ax, stat in zip(axf, DISPLAY):
        plot_delta_swarm(df, stat, ax=ax, p_value=res.get(stat, {}).get('p'),
                         group_col='group', value_col='value')
    for ax in axf[len(DISPLAY):]: fig.delaxes(ax)
    fig.suptitle(title, fontsize=13); fig.tight_layout()

In [ ]:
# §4a — within-phase effect (opto on vs off), WT vs HET
res_within, tbl_within = genotype_table(within_df)
swarm_grid(within_df, res_within, f'opto cohort · {DIST} · Δ opto (on − off), within-phase effect — WT vs HET   [sensitivity | bias]')
tbl_within

In [ ]:
# §4b — between-phase difference (opto vs masking, all trials), WT vs HET
if len(between_df):
    res_between, tbl_between = genotype_table(between_df)
    swarm_grid(between_df, res_between, f'opto cohort · {DIST} · Δ (opto − masking sessions, all trials) — between-phase difference — WT vs HET')
    display(tbl_between) if 'display' in dir() else print(tbl_between.to_string())
else:
    print('No between-session rows (masking missing).')

In [ ]:
# §4c — delta-of-deltas (silencing beyond artifact), WT vs HET
if len(delta_of_deltas_df):
    res_delta_of_deltas, tbl_delta_of_deltas = genotype_table(delta_of_deltas_df)
    swarm_grid(delta_of_deltas_df, res_delta_of_deltas, f'opto cohort · {DIST} · silencing isolated (delta-of-deltas) — WT vs HET')
    display(tbl_delta_of_deltas) if 'display' in dir() else print(tbl_delta_of_deltas.to_string())
else:
    print('No interaction rows (masking missing).')

In [ ]:
# §4d — genotype-mean update matrices ({het,wt} x {opto,masking} on/off)
def cond_ums(ids, session_type, trial_type):
    out = []
    for a in ids:
        ph = select_sessions(experiment.get_animal(a), distribution=DIST, session_type=session_type)
        c = filter_trials(ph, trial_type=trial_type)
        if c: out.append(compute_um(c))
    return out

for stype in ['opto', 'masking']:
    fig, axes = plt.subplots(2, 2, figsize=(8, 8), squeeze=False)
    for row, g in enumerate(['het', 'wt']):
        ids = groups.get(g, [])
        for col, tt in enumerate(['opto', 'non_opto']):
            ums = cond_ums(ids, stype, tt)
            if ums: plot_um(average_um(ums, min_sources=2), ax=axes[row, col])
            axes[row, col].set_title(f"{g} {stype} {'on' if tt=='opto' else 'off'} (n={len(ids)})", fontsize=9)
    fig.suptitle(f"opto cohort · {DIST} · {stype} genotype-mean UM (on/off)", fontsize=13); fig.tight_layout()

## §5 · Adaptation − rolling within-session trajectories

Rolling `ADAPT_STAT` over a 50-trial window within each **{DIST}** session,
opto vs masking. μ (PSE) reuses `compute_adaptation_per_session` and draws the
normative target (σ from masking-uniform non-opto trials); any other scalar
stat uses a rolling `fit_summary_stats` window. Set `ADAPT_STAT` in the next cell.

In [ ]:
# ── Adaptation helpers (inline): per-session rolling trajectories ─────────────
from collections import Counter, defaultdict
from behav_utils.analysis import compute_rolling_stats
import warnings

ADAPT_STAT = 'mu'   # SCALAR stat: 'mu' (PSE, normative line) | 'accuracy' | 'side_bias' |
                    # 'hard_accuracy' | 'win_stay' ...  (win_stay/recency noisy at 50-trial window)

PHASE_COLOUR = {'opto': '#1f77b4', 'masking': '#7f7f7f'}
GENO_COLOUR = {'het': '#d62728', 'wt': '#2ca02c'}


def is_scalar_stat(stat):
    """True if ``stat`` is a registered stat resolving to a single scalar."""
    try:
        from behav_utils.analysis.summary_stats import (
            SUMMARY_REGISTRY, get_stat_names_expanded)
        return any(stat in get_stat_names_expanded([f]) for f in SUMMARY_REGISTRY)
    except Exception:
        return False


def session_type_counts(experiment, aids, distribution):
    """Diagnostic: {animal: {session_type: n}} at a distribution — shows whether
    masking sessions exist yet for each animal."""
    out = {}
    for a in aids:
        try:
            ss = select_sessions(experiment.get_animal(a), distribution=distribution)
        except Exception:
            ss = []
        out[a] = dict(Counter(getattr(s, 'session_type', '?') for s in ss))
    return out


def session_curves(animal, distribution, stat, window=50, step=10, trials='all'):
    """Rolling ``stat`` trajectories for one animal's sessions at ``distribution``.

    Never raises: on any failure (no sessions, μ baseline/σ unresolvable, …) it
    returns empty sessions and warns, so a cohort loop is not killed by one animal.
    Returns ``{'sessions': [...], 'normative': float|nan, 'baseline0': bool}``,
    each session ``{'session_id', 'session_type', 'trials', 'values'}`` with a
    flat ``values`` array (this stat).

    μ keeps its own raw path (``compute_adaptation_per_session`` — raw PSE so the
    early, still-shallow windows survive, plus the normative reference line).
    Every other stat goes through the generic guarded ``compute_rolling_stats``;
    sessions are pre-filtered first (pipeline: select → filter → compute).
    """
    try:
        if stat == 'mu':
            sig = resolve_sigma(animal, source='uniform_masking', trials='non_opto')
            r = compute_adaptation_per_session(animal, distribution, sigma_percep=sig,
                                               window=window, step=step, trials=trials)
            norm = r.get('pse_normative', np.nan) - r.get('pse_expert', np.nan)
            segs = [{'session_id': e['session_id'], 'session_type': e.get('session_type', ''),
                     'trials': np.asarray(e['trials'], float),
                     'values': np.asarray(e['values'], float)}
                    for e in r.get('sessions', [])]
            return {'sessions': segs, 'normative': float(norm), 'baseline0': True}
        sessions = [s for s in select_sessions(animal, distribution=distribution,
                                                exclude_masking=False)
                    if getattr(s, 'session_type', '') in PHASE_COLOUR]
        sessions = filter_trials(sessions, trial_type=trials)
        r = compute_rolling_stats(sessions, stat_names=[stat], mode='per_session',
                                  window=window, step=step)
        segs = [{'session_id': e['session_id'], 'session_type': e['session_type'],
                 'trials': e['trials'], 'values': e['values'][stat]}
                for e in r['sessions']]
        return {'sessions': segs, 'normative': np.nan, 'baseline0': False}
    except Exception as exc:
        warnings.warn(f"{getattr(animal, 'animal_id', '?')}: adaptation "
                      f"({stat}, {distribution}) failed: {exc}")
        return {'sessions': [], 'normative': np.nan, 'baseline0': (stat == 'mu')}


def _mean_curve(segs):
    """Mean over sessions at each window-centre (centres align across sessions)."""
    acc = defaultdict(list)
    for e in segs:
        for x, y in zip(e['trials'], e['values']):
            if np.isfinite(y):
                acc[float(x)].append(y)
    if not acc:
        return None, None
    xs = np.array(sorted(acc))
    return xs, np.array([np.mean(acc[x]) for x in xs])


def _ylabel(stat, baseline0):
    return f'rolling {stat}' + (' (\u03bc \u2212 expert)' if baseline0 else '')


def _finish(ax, has_data, base0, norm, stat, empty_msg):
    if not has_data and empty_msg:
        ax.text(0.5, 0.5, empty_msg, transform=ax.transAxes, ha='center', va='center',
                fontsize=10, color='0.5')
    if base0:
        ax.axhline(0.0, color='0.6', lw=1, zorder=1)
    if np.isfinite(norm):
        ax.axhline(norm, color='crimson', ls='--', lw=1, zorder=1, label='normative')
    ax.set_xlabel('trial in session')
    ax.set_ylabel(_ylabel(stat, base0))
    if ax.get_legend_handles_labels()[0]:
        ax.legend(frameon=False, fontsize=8)


def plot_animal_adaptation(cur, stat, ax=None, title=None):
    """Single animal: thin per-session curves + thick per-phase mean, opto vs masking."""
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 5))
    has = False
    for phase, col in PHASE_COLOUR.items():
        segs = [e for e in cur['sessions'] if e['session_type'] == phase]
        for e in segs:
            ax.plot(e['trials'], e['values'], color=col, lw=0.8, alpha=0.35, zorder=2)
        mx, my = _mean_curve(segs)
        if mx is not None:
            ax.plot(mx, my, color=col, lw=2.6, zorder=4, label=f'{phase} (n={len(segs)})')
            has = True
    _finish(ax, has, cur.get('baseline0'), cur.get('normative', np.nan), stat,
            'no opto/masking sessions')
    if title:
        ax.set_title(title, fontsize=10)
    return ax


def plot_group_adaptation_2x2(curves_by_animal, by_animal, stat, title=None):
    """2x2: HET | WT (opto vs masking) on top, masking | opto (HET vs WT) on bottom."""
    aids = list(curves_by_animal)
    het = [a for a in aids if by_animal.get(a) == 'het']
    wt = [a for a in aids if by_animal.get(a) == 'wt']
    norm = next((curves_by_animal[a]['normative'] for a in aids
                 if np.isfinite(curves_by_animal[a].get('normative', np.nan))), np.nan)
    base0 = any(curves_by_animal[a].get('baseline0') for a in aids)

    def segs_of(group_aids, phase):
        return [e for a in group_aids for e in curves_by_animal[a]['sessions']
                if e['session_type'] == phase]

    def panel(ax, series, empty_msg):
        has = False
        for label, segs, col in series:
            for e in segs:
                ax.plot(e['trials'], e['values'], color=col, lw=0.5, alpha=0.2, zorder=2)
            mx, my = _mean_curve(segs)
            if mx is not None:
                ax.plot(mx, my, color=col, lw=2.4, zorder=4, label=f'{label} (n={len(segs)})')
                has = True
        _finish(ax, has, base0, norm, stat, empty_msg)

    fig, ax = plt.subplots(2, 2, figsize=(13, 10))
    panel(ax[0, 0], [(p, segs_of(het, p), PHASE_COLOUR[p]) for p in PHASE_COLOUR],
          'no HET sessions')
    ax[0, 0].set_title(f'HET (n={len(het)}) \u00b7 opto vs masking', fontsize=10)
    panel(ax[0, 1], [(p, segs_of(wt, p), PHASE_COLOUR[p]) for p in PHASE_COLOUR],
          'no WT sessions')
    ax[0, 1].set_title(f'WT (n={len(wt)}) \u00b7 opto vs masking', fontsize=10)
    panel(ax[1, 0], [('het', segs_of(het, 'masking'), GENO_COLOUR['het']),
                     ('wt', segs_of(wt, 'masking'), GENO_COLOUR['wt'])],
          'no masking sessions')
    ax[1, 0].set_title('masking \u00b7 HET vs WT', fontsize=10)
    panel(ax[1, 1], [('het', segs_of(het, 'opto'), GENO_COLOUR['het']),
                     ('wt', segs_of(wt, 'opto'), GENO_COLOUR['wt'])],
          'no opto sessions')
    ax[1, 1].set_title('opto \u00b7 HET vs WT', fontsize=10)
    if title:
        fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    return fig

In [ ]:
# diagnostic + single animal. Masking blocks come LATER in the curriculum, so
# many animals may have opto but no masking sessions yet at a hard distribution.
if not is_scalar_stat(ADAPT_STAT):
    raise ValueError(f"ADAPT_STAT={ADAPT_STAT!r} is not a scalar stat; use "
                     f"'mu','accuracy','side_bias','hard_accuracy','win_stay', ...")
print(f'{DIST} session types per animal:')
for a, c in session_type_counts(experiment, opto_ids, DIST).items():
    print(f'  {a} ({by_animal.get(a)}): {c or "none"}')

AID = next((a for a in opto_ids
            if select_sessions(experiment.get_animal(a), distribution=DIST, session_type='opto')),
           opto_ids[0] if opto_ids else None)
AID= 
if AID is not None:
    cur = session_curves(experiment.get_animal(AID), DIST, ADAPT_STAT, window=50, step=10)
    plot_animal_adaptation(cur, ADAPT_STAT,
        title=f"{AID} \u00b7 {by_animal.get(AID)} \u00b7 {DIST} \u00b7 rolling {ADAPT_STAT} (opto vs masking)")

In [ ]:
# 2x2 group view, this DIST  (empty phases annotate rather than break)
cba = {a: session_curves(experiment.get_animal(a), DIST, ADAPT_STAT, window=50, step=10)
       for a in opto_ids}
cba = {a: c for a, c in cba.items() if c['sessions']}
if cba:
    plot_group_adaptation_2x2(cba, by_animal, ADAPT_STAT,
        title=f"{DIST} \u00b7 rolling {ADAPT_STAT} adaptation (window=50)")
else:
    print(f'no opto/masking {DIST} sessions in cohort')